# AI-Powered Software QA Test Case Generator
**Graduation Project**


## Project Overview

This project builds an AI system that helps a Software QA Engineer generate test cases automatically from a software requirement, instead of writing them all by hand.

**Input:** A plain-text software requirement (for example: *"The user should be able to log in using a valid username and password."*)

**Output:** A structured set of test cases (positive, negative, and edge cases), a coverage/gap analysis of those test cases, and any genuinely missing test cases, generated and validated automatically.

**Main idea:** Use an open-source Large Language Model (Mistral-7B-Instruct) together with Retrieval-Augmented Generation (RAG) so the model's answers are grounded in a real QA reference document, not just what the model already "knows".


## Problem Statement

Writing test cases manually is slow and repetitive. A QA Engineer has to read a requirement, think through every positive, negative, and edge scenario, and make sure nothing important is missed. This takes time and still depends on the tester's own experience, so coverage gaps happen.

## Project Objective

Build a pipeline that takes a plain software requirement as input and automatically:
1. Generates a first set of test cases.
2. Reviews those test cases for coverage gaps.
3. Generates only the test cases that are genuinely missing.
4. Validates the missing test cases so nothing out-of-scope is added.
5. Returns a final, clean list of test cases ready to use.


## 1. Setup & Installation

In [1]:
# Install all required libraries in one place.
# (Combined into a single cell instead of being scattered across the notebook.)
!pip install -U langchain langchain-core langchain-community langchain-huggingface \
    langchain-text-splitters pypdf faiss-cpu sentence-transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 61.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**What this does:** installs every library the project needs — LangChain (to build the RAG pipeline), `pypdf` (to read the reference PDF), `faiss-cpu` (to store and search embeddings), `sentence-transformers` (to create embeddings), and `gradio` (for the demo GUI at the end of the notebook).

**Why we need it:** Kaggle environments don't always have every package pre-installed, and versions can change. Running this once at the top keeps all installation in a single place instead of scattered across the notebook.


## 2. Imports

In [2]:
import os
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

from pydantic import BaseModel, Field
from typing import List

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import gradio as gr

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/tmp/ipykernel_58/373204741.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


**What this does:** imports every library used later in the notebook — PyTorch and Transformers to load and run the LLM, Pydantic for the structured test case schema, LangChain pieces for the RAG pipeline, and Gradio for the GUI.

**Why we need it:** all imports are grouped here so the reader can see everything the project depends on at a glance, and so nothing is imported halfway through the notebook after it's already needed.

**Expected output:** the installed PyTorch version, whether a GPU is available, and the GPU name (this project needs a GPU — a T4 on Kaggle — because the model has 7 billion parameters).


## 3. Configuration

All the settings used across the notebook are collected here so they are easy to change in one place.


In [4]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Path to the QA reference PDF used for the RAG knowledge base
PDF_PATH = "/kaggle/input/datasets/alimoabdelsalam/graduation-project-pdf/Graduation Project PDF.pdf"

# Safety limit on how many tokens we feed into the model at once.
# This prevents "CUDA out of memory" / context-overflow errors when a prompt
# (requirement + retrieved QA knowledge) becomes long.
MAX_INPUT_TOKENS = 4096

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

**What this does:** defines the model name, the path to the reference PDF, a token limit, and the embedding model name as constants.

**Why we need it:** if any of these need to change later (a different model, a different PDF, a different embedding model), there is exactly one place to change them instead of hunting through the whole notebook.

**Note:** `PDF_PATH` points to a Kaggle Dataset path. If this notebook is run outside Kaggle, update this path to wherever the PDF is stored.


## 4. Hugging Face Login

Mistral-7B-Instruct is a gated model, so downloading it requires a Hugging Face account and access token.

**Security note:** never write your Hugging Face access token directly in a notebook cell (even commented out) — tokens can leak if the notebook is ever shared or made public. Run the cell below and paste your token into the secure prompt that appears; it will not be shown or saved in the notebook.


In [ ]:
# Log in to Hugging Face (needed to download the gated Mistral model).
# This opens a secure widget — paste your token there, do not hardcode it here.
login()

## 5. Load Model and Tokenizer

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,      # use half precision to reduce GPU memory usage
    device_map="auto"         # automatically place the model on GPU if available, else CPU
)

print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded on: cuda:0


**What this does:** downloads and loads the Mistral-7B-Instruct model and its tokenizer.

**Why we need it:** the tokenizer turns text into the numeric tokens the model understands (and back again), and the model itself is what actually generates the test cases. `dtype=torch.float16` loads the model in half precision so it fits comfortably on a single T4 GPU.

**Expected output:** `Model loaded on: cuda:0` (or `cpu` if no GPU is available, though generation will be very slow on CPU).


## 6. Text Generation Utilities

Two helper functions are used throughout the notebook:

- `generate_text`: general-purpose generation, returns the full decoded text (prompt + answer). Useful for a quick look at raw model output.
- `generate_structured_text`: returns only the newly generated answer, with the input prompt sliced off. This is used everywhere the output needs to be clean — either because it will be parsed (e.g. into the Pydantic schema) or because it will be shown directly to a user in the GUI or fed into the next step of the pipeline.


In [8]:
def generate_text(prompt, max_new_tokens=1000):
    """Generate text from a prompt and return the full decoded output (prompt + answer)."""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [9]:
def generate_structured_text(prompt, max_new_tokens=1000):
    """Generate text and return ONLY the newly generated tokens (no prompt echo).
    Used whenever the response needs to be clean: parsed into a schema, shown in the
    GUI, or passed as input into the next step of the pipeline."""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.3,   # lower temperature -> more consistent, easier-to-parse output
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    # Slice off the input tokens so only the generated answer is decoded
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

## 7. Approach 1 — Basic Prompt-Based Test Case Generation

As a first experiment, a simple prompt asks the model to generate test cases directly from a requirement, with no structure enforced on the output. This is intentionally shown using `generate_text`, so you can see the model's raw output including the echoed prompt — this is exactly why `generate_structured_text` was built next.


In [10]:
requirement = "The user should be able to log in using a valid username and password."

basic_prompt = f"""
You are a professional Software QA Engineer.

Analyze the following software requirement and generate test cases.

Requirement:
{requirement}

Generate:
1. Test Case ID
2. Test Case Description
3. Preconditions
4. Test Steps
5. Test Data
6. Expected Result
7. Test Type (Positive, Negative, or Edge Case)

Answer:
"""

basic_result = generate_text(basic_prompt)
print(basic_result)


You are a professional Software QA Engineer.

Analyze the following software requirement and generate test cases.

Requirement:
The user should be able to log in using a valid username and password.

Generate:
1. Test Case ID
2. Test Case Description
3. Preconditions
4. Test Steps
5. Test Data
6. Expected Result
7. Test Type (Positive, Negative, or Edge Case)

Answer:

1. Test Case ID: TC_001_Login_Valid_Credentials
2. Test Case Description: Verify that the user is able to log in successfully using valid username and password.
3. Preconditions: User has registered with valid credentials and the application is running.
4. Test Steps:
   a. Navigate to the login page.
   b. Enter the valid username in the username field.
   c. Enter the valid password in the password field.
   d. Click on the 'Login' button.
5. Test Data:
   a. Valid username: testuser1@example.com
   b. Valid password: Test@12345
6. Expected Result: The user is logged in successfully and redirected to the dashboard or 

## 8. Approach 2 — Structured Test Case Generation (Pydantic Schema)

The basic approach above returns free text, which is hard to process automatically. This section defines a schema with Pydantic and asks the model to return output that matches it, so the test cases can be parsed into Python objects instead of raw text.


In [11]:
class TestCase(BaseModel):
    test_case_id: str = Field(description="Unique ID of the test case")
    description: str = Field(description="Description of the test case")
    preconditions: str = Field(description="Preconditions required before executing the test")
    test_steps: List[str] = Field(description="Steps required to execute the test")
    test_data: str = Field(description="Test data used in the test case")
    expected_result: str = Field(description="Expected result of the test case")
    test_type: str = Field(description="Test type: Positive, Negative, or Edge Case")


class TestCaseResponse(BaseModel):
    test_cases: List[TestCase]


parser = PydanticOutputParser(pydantic_object=TestCaseResponse)
format_instructions = parser.get_format_instructions()

In [12]:
qa_prompt = PromptTemplate(
    template="""
You are a professional Software QA Engineer.

Analyze the following software requirement and generate comprehensive test cases.

Requirement:
{requirement}

For each test case, provide:
- Test Case ID
- Description
- Preconditions
- Test Steps
- Test Data
- Expected Result
- Test Type

Make sure to include:
- Positive test cases
- Negative test cases
- Edge cases

{format_instructions}
""",
    input_variables=["requirement"],
    partial_variables={"format_instructions": format_instructions}
)

In [13]:
def generate_qa_test_cases(requirement):
    prompt = qa_prompt.format(requirement=requirement)
    raw_response = generate_structured_text(prompt, max_new_tokens=1000)

    try:
        return parser.parse(raw_response)
    except Exception as e:
        print("Parsing Error:", e)
        print("Raw Response:")
        print(raw_response)
        return None


structured_result = generate_qa_test_cases(requirement)

if structured_result:
    for tc in structured_result.test_cases:
        print("=" * 50)
        print("ID:", tc.test_case_id)
        print("Description:", tc.description)
        print("Preconditions:", tc.preconditions)
        print("Steps:", tc.test_steps)
        print("Test Data:", tc.test_data)
        print("Expected Result:", tc.expected_result)
        print("Test Type:", tc.test_type)

Parsing Error: Failed to parse TestCaseResponse from completion {"test_cases": [{"test_case_id": "TC001", "description": "Valid username and password", "preconditions": "User exists in the system with valid credentials", "test_steps": ["Navigate to the login page", "Enter valid username", "Enter valid password", "Click on the login button"], "test_data": "admin@example.com, password123", "expected_result": "Successful login", "test_type": "Positive"}, {"test_case_id": "TC002", "description": "Invalid username", "preconditions": "User does not exist in the system", "test_steps": ["Navigate to the login page", "Enter invalid username", "Enter valid password", "Click on the login button"], "test_data": "invaliduser@example.com", "expected_result": "Login failed: Invalid username", "test_type": "Negative"}, {"test_case_id": "TC003", "description": "Invalid password", "preconditions": "User exists in the system with valid username", "test_steps": ["Navigate to the login page", "Enter valid 

**What this does:** asks the model to return test cases in a strict JSON-like format matching the `TestCaseResponse` schema, then parses that output into real Python objects (`TestCase` instances) instead of raw text.

**Why we need it:** structured output is what makes the rest of the pipeline (coverage analysis, missing test case detection, and the GUI) possible to build reliably — free text is hard to check and re-use programmatically.


## 9. Building the RAG Pipeline

The two approaches above only rely on what the LLM already "knows". To ground the test case generation in a real QA reference document, this section builds a Retrieval-Augmented Generation (RAG) pipeline:

1. Load the QA reference PDF.
2. Split it into overlapping text chunks.
3. Turn each chunk into an embedding (a numeric vector).
4. Store the embeddings in a FAISS vector database.
5. Create a retriever that returns the most relevant chunks for a given requirement.


In [14]:
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print("Number of pages loaded:", len(documents))

Number of pages loaded: 41


In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print("Number of chunks:", len(chunks))

Number of chunks: 99


In [16]:
embedding = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

vectordb = FAISS.from_documents(chunks, embedding)
print("Vector database created successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created successfully.


In [17]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

def get_qa_knowledge(requirement):
    """Retrieve the most relevant chunks from the QA reference PDF for a given requirement."""
    docs = retriever.invoke(requirement)
    return "\n\n".join(doc.page_content for doc in docs)


# Quick check that retrieval works
sample_knowledge = get_qa_knowledge(requirement)
print(sample_knowledge)

Beginners Guide To Software Testing 
  
Page 30 
 
  
Error Guessing  
It is a test case design technique where the testers use their experience to guess the 
possible errors that might occur and design test cases accordingly  to uncover them.  
Using any or a combination of the above described test case design techniques; you 
can develop effective test cases. 
 
What is a Use Case? 
 
A use case describes the system’s behavior under various conditions as it responds to 
a request from one of the users. The user initiates an interaction with the system to 
accomplish some goal. Different sequences of behavior, or scenarios, can unfold, 
depending on the particular requests made and conditions surrounding the requests. 
The use case collects together those different scenarios.  
Use cases are popular largely because they tell coherent stories about how the 
system will behave in use. The users of the system get to see just what this new 
system will be and get to react early.

Beginner

**What this does:** loads the reference PDF, breaks it into small overlapping chunks, converts each chunk into an embedding vector, and stores all of them in a FAISS index so the most relevant chunks can be searched later.

**Why we need it:** this is what lets the model use real QA knowledge from a trusted document instead of only its own training data, which makes the generated test cases more grounded and reliable.

**Expected output:** the number of pages loaded from the PDF, the number of text chunks created, and a confirmation that the vector database was built.


## 10. RAG-Based Test Case Generation (Main Pipeline)

This is the main generation step used for the rest of the notebook: the requirement is combined with the retrieved QA knowledge and sent to the model. This function uses `generate_structured_text` so the output is clean (no repeated prompt), since this output is passed into the next steps.


In [18]:
def generate_rag_test_cases(requirement):
    qa_knowledge = get_qa_knowledge(requirement)

    prompt = f"""
You are a professional Software QA Engineer.

Use the QA knowledge provided below as guidance when analyzing the requirement
and generating test cases.

QA Knowledge:
{qa_knowledge}

Software Requirement:
{requirement}

Generate comprehensive test cases for the requirement.

Include:
- Positive test cases
- Negative test cases
- Edge cases

For each test case, provide:
- Test Case ID
- Description
- Preconditions
- Test Steps
- Test Data
- Expected Result
- Test Type

Return the answer clearly and accurately.
"""

    return generate_structured_text(prompt, max_new_tokens=2000)


generated_test_cases = generate_rag_test_cases(requirement)
print(generated_test_cases)

Test Cases:

Test Case ID: TC001
Description: Test valid login with correct username and password
Preconditions: User is on the login page
Test Steps:
1. Enter valid username in the username field
2. Enter valid password in the password field
3. Click on the login button
Test Data: Valid username and password
Expected Result: User is redirected to the dashboard page
Test Type: Functional, Verification

Test Case ID: TC002
Description: Test invalid login with incorrect username and password
Preconditions: User is on the login page
Test Steps:
1. Enter incorrect username in the username field
2. Enter incorrect password in the password field
3. Click on the login button
Test Data: Incorrect username and password
Expected Result: Error message is displayed and user is not redirected to the dashboard page
Test Type: Functional, Verification

Test Case ID: TC003
Description: Test empty login fields
Preconditions: User is on the login page
Test Steps:
1. Leave the username field empty
2. Lea

## 11. QA Coverage and Gap Analysis

Once test cases are generated, the model is asked to act as a reviewer and check its own output for coverage gaps. Strict rules are included in the prompt so the model does not report scenarios as "missing" when they are already covered, and does not suggest scenarios that are out of scope for the requirement.


In [19]:
def analyze_qa_coverage(requirement, generated_test_cases):
    prompt = f"""
You are a Senior Software QA Engineer performing a strict test coverage and gap analysis.

Requirement:
{requirement}

Generated Test Cases:
{generated_test_cases}

Your task is to review EVERY generated test case carefully and identify ONLY genuine missing scenarios.

STRICT RULES:

1. First, create an internal checklist of ALL scenarios already covered by the generated test cases.
2. Compare every potential missing scenario against ALL existing test cases.
3. A scenario is considered COVERED if an existing test case tests the same functional condition, even if the wording or test case ID is different.
4. NEVER report an existing or equivalent scenario as missing.
5. NEVER duplicate an existing test case.
6. Do not suggest a positive test case if the expected result is login failure.
7. Do not classify invalid inputs or empty inputs as positive test cases.
8. Empty username and empty password are negative or edge cases, NOT positive test cases.
9. Invalid username and invalid password are already covered if they exist in the generated test cases.
10. If a scenario is already covered, explicitly state that it is covered and DO NOT list it under missing scenarios.
11. Only identify scenarios that are genuinely absent from the generated test cases.
12. Only suggest scenarios directly relevant to the given requirement.
13. Features such as password reset, session timeout, account lockout, and password complexity should be marked OUT OF SCOPE unless they are explicitly mentioned in the requirement.

Analyze the following:

1. Requirements Covered
List the requirement scenarios that are already covered.

2. Missing Test Scenarios
List ONLY genuinely missing scenarios.
For each one, explain why it is missing.

3. Missing Positive Test Cases
List ONLY missing positive scenarios.
Do not include negative or edge cases here.

4. Missing Negative Test Cases
List ONLY missing negative scenarios.
Do not include scenarios already covered.

5. Missing Edge Cases
List ONLY genuinely missing edge cases.

6. Additional Test Cases
Suggest ONLY relevant additional test cases that are not already covered.

7. Out of Scope
List scenarios that may be useful for a real system but are NOT directly supported by the given requirement.

IMPORTANT:
Before finalizing your answer, verify every reported missing scenario against every generated test case to ensure it is not already covered.

Provide a clear and structured QA analysis.
"""

    return generate_structured_text(prompt, max_new_tokens=2500)


coverage_analysis = analyze_qa_coverage(requirement, generated_test_cases)
print(coverage_analysis)

1. Requirements Covered:
- Test valid login with correct username and password (TC001)
- Test invalid login with incorrect username and password (TC002)
- Test login with empty fields (TC003)
- Test login with special characters in username and password (TC004)
- Test login with username and password length restrictions (TC005 and TC006)
- Test login with empty password (TC007)
- Test login with expired password (TC008)
- Test login with locked account (TC009)
- Test login with disabled account (TC010)
- Test login with forgotten password (TC011)
- Test login with multiple failed attempts (TC012)
- Test login with session timeout (TC013)
- Test login with multiple users (TC014)
- Test login with concurrent users (TC015)
- Test login with slow network connection (TC016)
- Test login with browser cache enabled (TC017)
- Test login with browser cookies enabled (TC018)
- Test login with different browsers (TC019)
- Test login with different operating systems (TC020)

2. Missing Test Scenar

## 12. Generating Missing Test Cases

In [20]:
def generate_missing_test_cases(requirement, generated_test_cases, analysis):
    prompt = f"""
You are a Senior Software QA Engineer.

Your task is to generate ONLY the genuinely missing test cases identified
during the QA coverage and gap analysis.

Requirement:
{requirement}

Existing Generated Test Cases:
{generated_test_cases}

QA Coverage and Gap Analysis:
{analysis}

STRICT RULES:

1. Review all existing test cases carefully.
2. Generate ONLY test cases that are genuinely missing.
3. DO NOT duplicate any existing test case.
4. DO NOT generate scenarios that are already covered.
5. DO NOT generate unrelated features.
6. Each new test case must be directly relevant to the requirement.
7. Include Positive, Negative, or Edge Case only when genuinely missing.
8. If there are no genuine missing test cases, clearly state:
   "No additional test cases are required."

For each new test case, provide:

- Test Case ID
- Description
- Preconditions
- Test Steps
- Test Data
- Expected Result
- Test Type

Return only the missing test cases and nothing else.
"""

    return generate_structured_text(prompt, max_new_tokens=2000)


# Character-level truncation is used here as a simple, reliable way to keep the
# prompt within a reasonable size when passing earlier long outputs back into the model.
missing_test_cases = generate_missing_test_cases(
    requirement,
    generated_test_cases[:6000],
    coverage_analysis[:6000]
)

print(missing_test_cases)

Missing Test Cases:

Test Case ID: TC016
Description: Test login with a slow network connection
Preconditions: User is on the login page with a slow network connection
Test Steps:
1. Simulate a slow network connection
2. Enter valid username and password
3. Click on the login button
Test Data: Valid username and password
Expected Result: User is able to log in despite the slow network connection
Test Type: Functional, Performance

Test Case ID: TC017
Description: Test login with browser cache enabled
Preconditions: User has visited the login page before and has browser cache enabled
Test Steps:
1. Enter valid username and password in the cached fields
2. Click on the login button
Test Data: Valid username and password
Expected Result: User is redirected to the dashboard page without having to enter the credentials again
Test Type: Functional, Verification

Test Case ID: TC018
Description: Test login with browser cookies enabled
Preconditions: User has visited the login page before and 

## 13. Validating the Missing Test Cases

In [21]:
def validate_missing_test_cases(requirement, original_test_cases, missing_test_cases):
    prompt = f"""
You are a Senior Software QA Engineer.

Your task is to strictly validate proposed missing test cases.

Requirement:
{requirement}

Original Test Cases:
{original_test_cases}

Proposed Missing Test Cases:
{missing_test_cases}

STRICT VALIDATION RULES:

1. Review every proposed test case individually.

2. A test case is VALID ONLY if it tests a scenario that is:
   - Directly supported by the Requirement, OR
   - A direct logical variation of the exact requirement.

3. The requirement is:
   "{requirement}"

4. DO NOT generate or keep test cases related to:
   - Password complexity
   - Minimum/maximum password length
   - Username or password case sensitivity
   - Account lockout
   - Session timeout
   - Password reset
   - Multiple devices or browsers
   - Performance testing
   - Security testing
   - Registration
   - Any unrelated functionality

5. DO NOT assume requirements that are not explicitly stated
   (e.g. do not assume password complexity, length limits, or case sensitivity rules exist).

6. DO NOT duplicate an existing test case.

7. If a proposed test case is not clearly supported by the requirement, REMOVE it.

8. If no valid missing test cases remain, return exactly:
No additional test cases are required.

For every valid missing test case, provide:

Test Case ID:
Description:
Preconditions:
Test Steps:
Test Data:
Expected Result:
Test Type:

Return ONLY the final validated missing test cases.
Do not include explanations or analysis.
"""

    return generate_structured_text(prompt, max_new_tokens=2000)


validated_missing_test_cases = validate_missing_test_cases(
    requirement,
    generated_test_cases[:6000],
    missing_test_cases[:6000]
)

print(validated_missing_test_cases)

Test Case ID: TC025
Description: Test login with a username that contains special characters in the middle
Preconditions: User is on the login page
Test Steps:
1. Enter a username with special characters in the middle (@usernam3)
2. Enter valid password in the password field
3. Click on the login button
Test Data: Username with special characters in the middle and valid password
Expected Result: User is redirected to the dashboard page (if the system supports such usernames)
Test Type: Functional, Verification

Test Case ID: TC026
Description: Test login with a username that starts with a number
Preconditions: User is on the login page
Test Steps:
1. Enter a username that starts with a number (1usernam3)
2. Enter valid password in the password field
3. Click on the login button
Test Data: Username that starts with a number and valid password
Expected Result: User is redirected to the dashboard page (if the system supports such usernames)
Test Type: Functional, Verification

Test Case I

## 14. Filtering and Final Output

As a final safety net, a simple keyword filter removes any leftover out-of-scope lines before showing the final result. This is a lightweight rule-based check that complements the model's own validation step above.


In [23]:
def filter_missing_test_cases(missing_test_cases):
    invalid_keywords = [
        "case sensitivity", "case-sensitive", "password complexity",
        "minimum length", "maximum length", "too short", "too long",
        "account lockout", "lockout", "session timeout", "password reset",
        "multiple devices", "multiple browsers", "performance testing",
        "security testing"
    ]

    filtered_lines = [
        line for line in missing_test_cases.split("\n")
        if not any(keyword in line.lower() for keyword in invalid_keywords)
    ]

    return "\n".join(filtered_lines)


final_missing_test_cases = filter_missing_test_cases(validated_missing_test_cases)

print("FINAL GENERATED TEST CASES\n" + "=" * 50)
print(generated_test_cases)

print("\nFINAL MISSING TEST CASES (validated & filtered)\n" + "=" * 50)
print(final_missing_test_cases)

FINAL GENERATED TEST CASES
Test Cases:

Test Case ID: TC001
Description: Test valid login with correct username and password
Preconditions: User is on the login page
Test Steps:
1. Enter valid username in the username field
2. Enter valid password in the password field
3. Click on the login button
Test Data: Valid username and password
Expected Result: User is redirected to the dashboard page
Test Type: Functional, Verification

Test Case ID: TC002
Description: Test invalid login with incorrect username and password
Preconditions: User is on the login page
Test Steps:
1. Enter incorrect username in the username field
2. Enter incorrect password in the password field
3. Click on the login button
Test Data: Incorrect username and password
Expected Result: Error message is displayed and user is not redirected to the dashboard page
Test Type: Functional, Verification

Test Case ID: TC003
Description: Test empty login fields
Preconditions: User is on the login page
Test Steps:
1. Leave the 

## 15. Full Pipeline Function

Every step above was run manually, one cell at a time, for a single hardcoded `requirement`. To reuse the whole pipeline for **any** requirement (and to power the GUI in the next section), this section wraps all five steps into one function.

This does not add any new logic — it only combines the steps that already exist above into a single, reusable function.


In [28]:
def run_qa_test_case_pipeline(requirement):
    """Run the full pipeline for a given requirement and return every intermediate result.

    Steps: generate test cases (RAG) -> analyze coverage -> generate missing
    test cases -> validate them -> filter out-of-scope lines.
    """
    generated = generate_rag_test_cases(requirement)
    coverage = analyze_qa_coverage(requirement, generated)
    missing = generate_missing_test_cases(requirement, generated[:6000], coverage[:6000])
    validated = validate_missing_test_cases(requirement, generated[:6000], missing[:6000])
    final_missing = filter_missing_test_cases(validated)

    return {
        "test_cases": generated,
        "coverage_analysis": coverage,
        "missing_test_cases": final_missing,
    }

In [29]:
# ---- Helpers for the GUI: PDF requirement extraction + a simple result cache ----

# Caps how many requirements we process from one uploaded PDF, so a large
# requirements document doesn't turn into dozens of sequential model calls.
MAX_REQUIREMENTS_FROM_PDF = 5

def extract_requirements_from_pdf(pdf_file):
    """Load a user-uploaded PDF and pull out individual requirement lines.
    Reuses the same PyPDFLoader already used for the QA reference PDF —
    no new PDF library or logic is introduced."""
    pdf_path = pdf_file.name if hasattr(pdf_file, "name") else pdf_file
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    full_text = "\n".join(page.page_content for page in pages)

    candidate_lines = [line.strip() for line in full_text.split("\n")]
    # Keep lines that look like actual sentences, drop blank lines / page numbers / headers
    requirements = [
        line for line in candidate_lines
        if len(line) >= 15 and any(c.isalpha() for c in line)
    ]
    return requirements


# Simple in-memory cache: if the same requirement is submitted again
# (e.g. the user clicks Generate twice, or a PDF repeats a line), we
# reuse the previous result instead of re-running 4 model calls.
_pipeline_cache = {}

def run_qa_test_case_pipeline_cached(requirement):
    key = requirement.strip().lower()
    if key in _pipeline_cache:
        return _pipeline_cache[key]
    result = run_qa_test_case_pipeline(requirement)
    _pipeline_cache[key] = result
    return result

## 16. GUI — Interactive Demo

A simple Gradio interface for demonstrating the project. The user types a software requirement, clicks **Generate**, and sees the generated test cases, the coverage analysis, and the validated missing test cases in separate tabs.

Gradio was chosen because it runs directly inside a Kaggle notebook and produces a shareable link, which makes it realistic to demo live during the graduation discussion without deploying anything separately.


In [33]:
custom_css = """
.gradio-container {max-width: 900px !important; margin: auto;}
#title {text-align: center;}
"""

def run_pipeline_for_gui(mode, requirement_text, pdf_file, progress=gr.Progress()):
    if mode == "Upload PDF":
        if pdf_file is None:
            yield "Please upload a PDF file containing requirements.", "", "", ""
            return
        try:
            requirements = extract_requirements_from_pdf(pdf_file)
        except Exception as e:
            yield f"Could not read the uploaded PDF: {e}", "", "", ""
            return
        if not requirements:
            yield "No readable requirement text was found in the uploaded PDF.", "", "", ""
            return
    else:
        requirement_text = (requirement_text or "").strip()
        if not requirement_text:
            yield "Please enter a software requirement before generating test cases.", "", "", ""
            return
        requirements = [requirement_text]

    note = ""
    if len(requirements) > MAX_REQUIREMENTS_FROM_PDF:
        note = (f"Found {len(requirements)} requirements in the PDF — processing only the "
                f"first {MAX_REQUIREMENTS_FROM_PDF} to keep generation time reasonable.\n\n")
        requirements = requirements[:MAX_REQUIREMENTS_FROM_PDF]

    multiple = len(requirements) > 1
    all_test_cases, all_coverage, all_missing = [], [], []
    total = len(requirements)

    for i, req in enumerate(requirements, start=1):
        progress((i - 1) / total, desc=f"Processing requirement {i} of {total}...")
        header = f"=== Requirement {i}: {req} ===\n" if multiple else ""

        try:
            result = run_qa_test_case_pipeline_cached(req)
            all_test_cases.append(header + result["test_cases"])
            all_coverage.append(header + result["coverage_analysis"])
            all_missing.append(header + result["missing_test_cases"])
        except Exception as e:
            error_text = header + f"Something went wrong: {e}"
            all_test_cases.append(error_text)
            all_coverage.append(error_text)
            all_missing.append(error_text)

        status = note + (f"Processed {i} of {total} requirement(s)."
                          if i < total else f"Done — processed {total} requirement(s).")
        yield status, "\n\n".join(all_test_cases), "\n\n".join(all_coverage), "\n\n".join(all_missing)


def toggle_input_mode(mode):
    return gr.update(visible=(mode == "Manual Input")), gr.update(visible=(mode == "Upload PDF"))

with gr.Blocks(
    title="AI QA Test Case Generator",
    theme=gr.themes.Soft(primary_hue="blue"),
    css=custom_css
) as demo:

    gr.Markdown(
        "# 🧪 AI-Powered Software QA Test Case Generator",
        elem_id="title"
    )

    gr.Markdown(
        "Generate structured test cases from a software requirement — "
        "type one in directly, or upload a requirements PDF."
    )

    input_mode = gr.Radio(
        ["Manual Input", "Upload PDF"],
        value="Manual Input",
        label="Requirement Source"
    )

    with gr.Group(visible=True) as manual_group:
        requirement_input = gr.Textbox(
            label="📝 Software Requirement",
            placeholder=(
                "e.g. The user should be able to log in "
                "using a valid username and password."
            ),
            lines=3
        )

    with gr.Group(visible=False) as pdf_group:
        pdf_input = gr.File(
            label="📄 Upload Requirement PDF",
            file_types=[".pdf"],
            type="filepath"
        )

    generate_button = gr.Button(
        "🚀 Generate Test Cases",
        variant="primary"
    )

    status_output = gr.Markdown()

    with gr.Tabs():

        with gr.Tab("Generated Test Cases"):
            test_cases_output = gr.Textbox(
                label="Test Cases",
                lines=20
            )

        with gr.Tab("Coverage & Gap Analysis"):
            coverage_output = gr.Textbox(
                label="Coverage Analysis",
                lines=20
            )

        with gr.Tab("Validated Missing Test Cases"):
            missing_output = gr.Textbox(
                label="Missing Test Cases",
                lines=20
            )

    input_mode.change(
        fn=toggle_input_mode,
        inputs=input_mode,
        outputs=[manual_group, pdf_group]
    )

    generate_button.click(
        fn=run_pipeline_for_gui,
        inputs=[
            input_mode,
            requirement_input,
            pdf_input
        ],
        outputs=[
            status_output,
            test_cases_output,
            coverage_output,
            missing_output
        ]
    )

demo.queue()

demo.launch(
    share=True,
    debug=False
)

/tmp/ipykernel_58/3227034628.py:59: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://0743a0e55cdd08c9b4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 17. Conclusion

This project shows an end-to-end pipeline for AI-assisted software test case generation:

- An open-source LLM (Mistral-7B-Instruct) is used to read a plain-text requirement and produce test cases.
- A RAG pipeline grounds the generation in a real QA reference document instead of relying only on the model's built-in knowledge.
- A second pass reviews the generated test cases for coverage gaps.
- A third pass generates only the genuinely missing test cases, which are then validated and filtered against the original requirement.
- A Gradio GUI wraps the whole pipeline so it can be demonstrated interactively.
